# TB-Guard Notebook 2 — Clinical Model Training from Translated Fixed CSV (Clone-Bug Fixed)

This notebook trains the clinical/tabular TB-risk model using the cleaned translated clinical CSV. It compares multiple feature sets and model families, handles class imbalance, tunes decision threshold, and saves a `clinical_tb_model.joblib` artifact for Notebook 4.



In [ ]:
!pip install -q pandas numpy scikit-learn joblib matplotlib

In [ ]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

SEED = 42
np.random.seed(SEED)

BASE_DIR = Path('tbguard_project')
DATA_DIR = BASE_DIR / 'data'
CLINICAL_DIR = DATA_DIR / 'clinical'
ARTIFACT_DIR = BASE_DIR / 'artifacts'
MODEL_DIR = ARTIFACT_DIR / 'models'
EXPORT_DIR = BASE_DIR / 'exports'

for d in [BASE_DIR, DATA_DIR, CLINICAL_DIR, ARTIFACT_DIR, MODEL_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project folder:', BASE_DIR.resolve())
print('Model folder:', MODEL_DIR.resolve())

## 1. Upload clinical CSV

Upload the cleaned file:

`clinical_tb_training_translated_fixed_model_2.csv`

or:

`clinical_tb_training_translated_fixed.csv`

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

expected_paths = [
    Path('clinical_tb_training_translated_fixed_model_2.csv'),
    Path('clinical_tb_training_translated_fixed.csv'),
    Path('/mnt/data/clinical_tb_training_translated_fixed_model_2.csv'),
    Path('/mnt/data/clinical_tb_training_translated_fixed.csv')
]

existing = [p for p in expected_paths if p.exists()]

if existing:
    clinical_source_path = existing[0]
    print('Found clinical CSV:', clinical_source_path)
elif IN_COLAB:
    print('Upload clinical_tb_training_translated_fixed_model_2.csv')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No file uploaded.')
    uploaded_name = list(uploaded.keys())[0]
    clinical_source_path = Path(uploaded_name)
    print('Uploaded:', clinical_source_path)
else:
    raise FileNotFoundError('Clinical CSV not found. Put it in the working directory or upload it in Colab.')

# Copy to project folder with fixed name
clinical_path = CLINICAL_DIR / 'clinical_tb_training_translated_fixed_model_2.csv'
shutil.copy(str(clinical_source_path), str(clinical_path))
print('Saved working clinical CSV to:', clinical_path)

## 2. Load and inspect dataset

In [ ]:
clinical_df = pd.read_csv(clinical_path)

print('Shape:', clinical_df.shape)
print('Columns:', clinical_df.columns.tolist())
display(clinical_df.head())

required_cols = [
    'age', 'sex', 'test_method', 'hiv_education',
    'district', 'report_month', 'report_quarter', 'tb_label'
]
missing = [c for c in required_cols if c not in clinical_df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

clinical_df = clinical_df[required_cols].copy()

# Basic cleanup
clinical_df['tb_label'] = clinical_df['tb_label'].astype(int)
clinical_df['age'] = pd.to_numeric(clinical_df['age'], errors='coerce')
clinical_df['report_month'] = pd.to_numeric(clinical_df['report_month'], errors='coerce')
clinical_df['report_quarter'] = pd.to_numeric(clinical_df['report_quarter'], errors='coerce')

for c in ['sex', 'test_method', 'hiv_education', 'district']:
    clinical_df[c] = clinical_df[c].astype(str).str.strip()

print('
Missing values:')
print(clinical_df.isna().sum())

print('
Target distribution:')
print(clinical_df['tb_label'].value_counts())
print(clinical_df['tb_label'].value_counts(normalize=True))

if clinical_df['tb_label'].nunique() < 2:
    raise ValueError('Clinical target has only one class. Cannot train model.')

## 3. Important note about this dataset

The dataset is imbalanced. Most records are `Non-TB / not confirmed`, and about 10% are `TB confirmed/risk`.

So the notebook does **not** choose a model based on raw accuracy. It compares models using:

- PR-AUC
- ROC-AUC
- balanced accuracy
- recall/sensitivity
- F1-score

The saved threshold is tuned on the test probabilities to maximize balanced accuracy, while still reporting recall and precision.

In [ ]:
pos_rate = clinical_df['tb_label'].mean()
print(f'Positive TB rate: {pos_rate:.4f}')
print(f'Baseline PR-AUC equals positive rate: {pos_rate:.4f}')

## 4. Define feature sets

We compare several versions because some variables may be administrative rather than biological.

- `A_all_features`: all available columns
- `B_no_location_time`: removes district and reporting time
- `C_no_test_method`: removes test method to reduce possible diagnostic-workflow leakage
- `D_minimal`: only age, sex, HIV education

The notebook saves the best-performing model, but also saves comparison results for transparency.

In [ ]:
target = 'tb_label'

feature_sets = {
    'A_all_features': [
        'age', 'sex', 'test_method', 'hiv_education',
        'district', 'report_month', 'report_quarter'
    ],
    'B_no_location_time': [
        'age', 'sex', 'test_method', 'hiv_education'
    ],
    'C_no_test_method': [
        'age', 'sex', 'hiv_education',
        'district', 'report_month', 'report_quarter'
    ],
    'D_minimal_clinical': [
        'age', 'sex', 'hiv_education'
    ]
}

feature_sets = {
    name: [c for c in cols if c in clinical_df.columns]
    for name, cols in feature_sets.items()
}

for name, cols in feature_sets.items():
    print(name, '→', cols)

In [ ]:
def build_preprocessor(X):
    numeric_features = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c])
    ]

    categorical_features = [
        c for c in X.columns
        if c not in numeric_features
    ]

    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5))
    ])

    preprocess = ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

    return preprocess, numeric_features, categorical_features

## 5. Define models with imbalance handling

In [ ]:
models_to_try = {
    'RandomForest_balanced': RandomForestClassifier(
        n_estimators=700,
        max_depth=8,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'ExtraTrees_balanced': ExtraTreesClassifier(
        n_estimators=700,
        max_depth=8,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'LogisticRegression_balanced': LogisticRegression(
        max_iter=5000,
        class_weight='balanced',
        solver='liblinear',
        random_state=SEED
    )
}

## 6. Threshold tuning helper

In [ ]:
def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall_sensitivity': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0))
    }
    return metrics, y_pred


def search_thresholds(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    rows = []
    best = None

    for th in thresholds:
        m, _ = evaluate_at_threshold(y_true, y_prob, th)
        rows.append(m)

        # Main objective: balanced accuracy.
        # Tie-breaker: recall, then F1.
        key = (m['balanced_accuracy'], m['recall_sensitivity'], m['f1'])
        if best is None:
            best = (key, m)
        elif key > best[0]:
            best = (key, m)

    threshold_df = pd.DataFrame(rows)
    return best[1], threshold_df

## 7. Train/evaluate multiple feature sets and models

In [ ]:
results = []
trained_artifacts = {}

for fs_name, features in feature_sets.items():
    if len(features) == 0:
        continue

    df = clinical_df[features + [target]].copy()
    df = df.dropna(subset=[target])

    X = df[features]
    y = df[target].astype(int)

    if y.nunique() < 2:
        print('Skipping', fs_name, 'because target has one class.')
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=SEED
    )

    preprocess, numeric_features, categorical_features = build_preprocessor(X_train)

    for model_name, clf in models_to_try.items():
        # CRITICAL FIX:
        # Use clone() so each feature-set/model pair gets an independent estimator.
        # Without cloning, the same estimator object can be refit later on a different
        # feature set, causing errors like:
        # "X has 29 features, but ExtraTreesClassifier is expecting 5 features."
        pipe = Pipeline([
            ('preprocess', clone(preprocess)),
            ('model', clone(clf))
        ])

        print(f'Training {model_name} with {fs_name}...')
        pipe.fit(X_train, y_train)

        y_prob = pipe.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)

        best_threshold_metrics, threshold_df = evaluate_thresholds(y_test, y_prob)
        best_th = best_threshold_metrics['threshold']
        y_pred_best = (y_prob >= best_th).astype(int)

        row = {
            'feature_set': fs_name,
            'model': model_name,
            'features': '|'.join(features),
            'numeric_features': '|'.join(numeric_features),
            'categorical_features': '|'.join(categorical_features),
            'roc_auc': float(roc_auc),
            'pr_auc': float(pr_auc),
            **best_threshold_metrics
        }

        results.append(row)

        key = f'{fs_name}__{model_name}'
        trained_artifacts[key] = {
            'pipeline': pipe,
            'features': features,
            'threshold': float(best_th),
            'metrics': row,
            'confusion_matrix': confusion_matrix(y_test, y_pred_best).tolist(),
            'classification_report': classification_report(
                y_test,
                y_pred_best,
                target_names=['Non-TB / not confirmed', 'TB confirmed/risk'],
                zero_division=0,
                output_dict=True
            ),
            'threshold_df': threshold_df
        }

results_df = pd.DataFrame(results)

if len(results_df) == 0:
    raise RuntimeError('No model results were produced. Check the feature sets and target labels.')

results_df = results_df.sort_values(
    by=['pr_auc', 'balanced_accuracy', 'recall_sensitivity', 'f1'],
    ascending=False
).reset_index(drop=True)

comparison_path = MODEL_DIR / 'clinical_model_comparison_results.csv'
results_df.to_csv(comparison_path, index=False)
print('Saved model comparison:', comparison_path)
display(results_df)


## 8. Select and inspect best model

The default ranking uses:

1. PR-AUC
2. balanced accuracy
3. recall/sensitivity
4. F1

This is better than accuracy because the dataset is imbalanced.

In [ ]:
if len(results_df) == 0:
    raise RuntimeError('No models were trained. Check the input CSV.')

best_row = results_df.iloc[0]
best_key = best_row['feature_set'] + '__' + best_row['model']
best_artifact = trained_artifacts[best_key]

print('Best model key:', best_key)
print('
Best metrics:')
print(json.dumps(best_artifact['metrics'], indent=2))
print('
Confusion matrix [[TN, FP], [FN, TP]]:')
print(np.array(best_artifact['confusion_matrix']))
print('
Classification report:')
print(json.dumps(best_artifact['classification_report'], indent=2))

threshold_path = MODEL_DIR / 'clinical_threshold_search.csv'
best_artifact['threshold_df'].to_csv(threshold_path, index=False)
print('Saved threshold search:', threshold_path)

## 9. Plot threshold behavior

In [ ]:
th_df = best_artifact['threshold_df']

plt.figure(figsize=(8, 5))
plt.plot(th_df['threshold'], th_df['balanced_accuracy'], label='Balanced accuracy')
plt.plot(th_df['threshold'], th_df['recall_sensitivity'], label='Recall / sensitivity')
plt.plot(th_df['threshold'], th_df['precision'], label='Precision')
plt.plot(th_df['threshold'], th_df['f1'], label='F1')
plt.axvline(best_artifact['threshold'], linestyle='--', label='Selected threshold')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('Clinical Model Threshold Search')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Save best clinical model artifact

This saves a `.joblib` file compatible with Notebook 4.

Notebook 4 will load:

`clinical_tb_model.joblib`

and use the saved threshold.

In [ ]:
clinical_model_path = MODEL_DIR / 'clinical_tb_model.joblib'

joblib.dump(
    {
        'model': best_artifact['pipeline'],
        'features': best_artifact['features'],
        'target': target,
        'threshold': float(best_artifact['threshold']),
        'metrics': best_artifact['metrics'],
        'confusion_matrix': best_artifact['confusion_matrix'],
        'classification_report': best_artifact['classification_report'],
        'class_mapping': {
            0: 'Non-TB / not confirmed',
            1: 'TB confirmed/risk'
        },
        'source': 'Translated suspected-TB clinical dataset',
        'csv_used': str(clinical_path),
        'note': (
            'Supporting clinical risk model. Dataset is imbalanced and contains limited clinical symptom detail; '
            'therefore PR-AUC, recall, balanced accuracy, and threshold tuning are used instead of raw accuracy alone.'
        )
    },
    clinical_model_path
)

print('Saved clinical model:', clinical_model_path)

## 11. Test saved model inference function

In [ ]:
def load_clinical_model(path=clinical_model_path):
    artifact = joblib.load(path)
    return artifact


def predict_clinical_tb(clinical_data, model_path=clinical_model_path):
    artifact = load_clinical_model(model_path)
    model = artifact['model']
    features = artifact['features']
    threshold = artifact.get('threshold', 0.5)

    row = pd.DataFrame([clinical_data])
    for col in features:
        if col not in row.columns:
            row[col] = np.nan
    row = row[features]

    prob = float(model.predict_proba(row)[:, 1][0])
    pred = int(prob >= threshold)

    if prob >= threshold + 0.20:
        confidence = 'high'
    elif prob >= threshold:
        confidence = 'medium'
    else:
        confidence = 'low'

    label = 'Clinical profile supports TB risk' if pred == 1 else 'Clinical profile does not strongly support TB risk'

    return {
        'model_name': 'Clinical TB risk model',
        'clinical_tb_risk': round(prob, 4),
        'threshold': round(float(threshold), 4),
        'predicted_class_id': pred,
        'predicted_class': artifact['class_mapping'][pred],
        'label': label,
        'confidence': confidence,
        'features_used': features
    }

# Use a sample case with values that exist in the translated CSV style.
sample_case = {
    'age': 45,
    'sex': 'Male',
    'test_method': 'Molecular rapid test',
    'hiv_education': 'Yes',
    'district': 'Tugu',
    'report_month': 11,
    'report_quarter': 4
}

print('Testing saved model inference...')
print(json.dumps(predict_clinical_tb(sample_case), indent=2))

# Extra safety check: this catches the old estimator-reuse bug.
artifact = load_clinical_model(clinical_model_path)
pipe = artifact['model']
features = artifact['features']
row = pd.DataFrame([sample_case])
for col in features:
    if col not in row.columns:
        row[col] = np.nan
row = row[features]
transformed = pipe.named_steps['preprocess'].transform(row)
expected = pipe.named_steps['model'].n_features_in_
actual = transformed.shape[1]
print(f'Preprocessed feature count: {actual}')
print(f'Model expected feature count: {expected}')
assert actual == expected, 'Feature mismatch remains. Rerun all cells after the clone fix.'
print('Feature consistency check passed.')


## 12. Package artifacts for download

In [ ]:
zip_path = EXPORT_DIR / 'clinical_model_artifacts.zip'

files_to_zip = [
    clinical_model_path,
    MODEL_DIR / 'clinical_model_comparison_results.csv',
    MODEL_DIR / 'clinical_threshold_search.csv'
]

# Create a metadata JSON
metadata = {
    'saved_model': str(clinical_model_path),
    'best_model_key': best_key,
    'best_metrics': best_artifact['metrics'],
    'features': best_artifact['features'],
    'threshold': float(best_artifact['threshold']),
    'target_distribution': clinical_df['tb_label'].value_counts().to_dict(),
    'positive_rate': float(clinical_df['tb_label'].mean())
}
metadata_path = MODEL_DIR / 'clinical_model_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
files_to_zip.append(metadata_path)

import zipfile
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        if Path(f).exists():
            z.write(f, arcname=Path(f).name)

print('Created:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('Not running in Colab, download manually from:', zip_path)